# Chapter 9 Running Interactive Scripts

## 1. Installing DukPy

In [2]:
import dukpy

dukpy.evaljs("2 + 2")

4

In [ ]:
# 책 내용과 달리 dukpy 내부 엔진이 quickjs로 바뀌면서 최신 문법 지원되는 것으로 보임
try:
    print(dukpy.evaljs("let a = 4.2; a * 2"))
except e:
    print("ES6 error", e)

8.4


## 2. Running JavaScript Code

In [ ]:
import os
from browser import URL, Tab

with open("test.html", "w") as f:
    f.write('<script src="test.js"></script>')

with open("test.js", "w") as f:
    f.write("var x = 2;\nx + x;")

tab = Tab(600)
tab.load(URL("file://" + os.path.abspath("test.html")))

os.remove("test.html")
os.remove("test.js")

### 💡 `<script>` 실행 방식 비교 (Blocking vs defer vs async vs nomodule)

> 실제 브라우저는 `<script>` 태그를 만나는 즉시 HTML 파싱을 중단(Blocking)하고 스크립트를 다운로드/실행하지만, 우리 브라우저는 HTML 트리를 다 만든 뒤 실행하므로 `defer`와 유사하게 동작합니다.

| 방식 | 다운로드 | 실행 타이밍 | 파서 차단 | 실행 순서 보장 | 주 용도 |
| :--- | :--- | :--- | :---: | :---: | :--- |
| **기본 (Blocking)** | 즉시 (동기) | 다운로드 직후 | **O** | **O** | 기본 동작 (구형 방식) |
| **`defer`** | 백그라운드 병렬 | **HTML 파싱 완료 직후** (`DOMContentLoaded` 전) | X | **O** | DOM 제어 및 메인 앱 로직 (권장) |
| **`async`** | 백그라운드 병렬 | **다운로드 완료 즉시** | 실행 시 O | **X** (다운로드 순) | 독립 트래커, 분석 도구 (GA 등) |
| **`nomodule`** | 구형 브라우저만 | 구형 브라우저에서만 | - | **O** | 구형 브라우저용 폴백 (`type="module"`과 쌍으로 사용) |

---

#### 🔍 Speculative Parsing (추측 파싱 / Preload Scanner)
- **개념**: 기본 Blocking 스크립트 실행으로 메인 파서가 멈춰 있을 때, 백그라운드 보조 파서가 뒤쪽 HTML을 미리 훑어 외부 리소스(CSS, 이미지, 후속 JS)를 사전 다운로드하는 최적화 기법.
- **원리**: 선행 스크립트가 `document.write` 등으로 HTML 구조를 뒤엎지 않을 것이라 "추측(Speculate)"하여 네트워크 병목(Waterfall)을 방지. (추측 실패 시 임시 상태 롤백/폐기)
- **브라우저 구현**: 멀티스레드 동기화와 롤백 복잡성 때문에, 본 실습 브라우저는 전체 파싱 후 일괄 실행(`defer` 의미론)으로 단순화함.

---

#### 🌐 현대 웹(SPA vs SSR)과의 연관성 및 프레임워크별 해결 전략
- **SPA (CSR)**: 번들러가 `type="module"`/`defer`를 자동 적용해 파서 블로킹은 없으나, HTML이 껍데기(`<div id="root"></div>`)뿐이라 Speculative Parser가 리소스를 미리 찾지 못하고 거대 번들 로드 전까지 빈 화면(흰 화면 병목) 발생.
- **SSR**: 완성된 HTML을 주므로 Speculative Parser가 리소스를 초고속 사전 로딩함. 단, 화면은 보이지만 JS가 로드되어 이벤트 리스너를 결합하기 전까지 동작하지 않는 **하이드레이션(Hydration) 병목**이 현대의 주요 과제로 등장.
- **Vue / Nuxt**: 컴파일러 정적 호이스팅(불변 VNode 하이드레이션 제외), 비동기 컴포넌트(`defineAsyncComponent`), 뷰포트 기반 지연 하이드레이션(Lazy Hydration).
- **Angular**: 비파괴적 하이드레이션(DOM 재사용), `@defer` 블록(뷰포트/인터랙션 조건부 지연 로딩), 이벤트 리플레이(하이드레이션 전 클릭 이벤트 캡처 후 재생).
- **React / Next.js**: 스트리밍 SSR + 선택적 하이드레이션(Suspense 경계별 점진 하이드레이션), 서버 컴포넌트(RSC, 클라이언트에 JS 0바이트 전송).
- **신흥 아키텍처 (Astro / Qwik)**: 아일랜드 아키텍처(필요한 컴포넌트만 독립 하이드레이션), Resumability(서버 상태 직렬화로 하이드레이션 자체를 생략).

## 3. Exporting Functions

In [ ]:
import os
from browser import URL, Tab

# 1. a.js에서 변수 선언, b.js에서 해당 변수를 읽고 console.log()로 출력
with open("a.js", "w") as f:
    f.write("var x = 20;")

with open("b.js", "w") as f:
    f.write('console.log("Hi from JS! x + x = " + (x + x));')

with open("page.html", "w") as f:
    f.write('<script src="a.js"></script><script src="b.js"></script>')

# 2. 브라우저 탭에서 로드하여 상태 공유 및 console.log 동작 검증
tab = Tab(600)
tab.load(URL("file://" + os.path.abspath("page.html")))

# 3. 임시 파일 정리
os.remove("a.js")
os.remove("b.js")
os.remove("page.html")

### 💡 [참고] 발표 & 심화 노트: 4계층 아키텍처와 Host Bindings vs Polyfill

#### 1. 파이썬-자바스크립트 간 4계층 호출 아키텍처
브라우저가 파이썬 함수(`print`)를 `export_function("log", print)`로 내보내면, JS 엔진의 C 브리지를 거쳐 웹 표준 `console.log`로 래핑됨.

```text
[1] 웹 스크립트         : console.log("Hello") 호출
         ↓
[2] 브라우저 JS 런타임   : runtime.js (call_python("log", "Hello") 래핑)
         ↓ (JSON 직렬화 및 C-FFI 바운더리)
[3] C 언어 JS 엔진      : DukPy C 브리지 (수출된 함수 맵에서 "log" 탐색)
         ↓
[4] 브라우저 호스트      : Python의 print("Hello") 실행
```

#### 2. Host Bindings(내장 바인딩) vs Polyfill(폴리필)
- **호스트 바인딩 (Builtins)**: 브라우저/런타임 개발자가 C++/Python 등 호스트 언어의 저수준 기능을 웹 표준 API 형태로 포장해 엔진에 주입하는 공식 브리지 코드 (우리의 `runtime.js`).
- **폴리필 (Polyfill)**: 웹 서비스 개발자가 구형 브라우저에서 누락된 표준 스펙을 메우기 위해 웹페이지 레벨에서 끼워 넣는 하위 호환성 패치 코드 (`core-js` 등).

#### 3. 실제 브라우저 및 런타임 소스코드 사례
- **WebKit (Safari)**: 엔진 내부 [`Source/JavaScriptCore/builtins/`](https://github.com/WebKit/WebKit/tree/main/Source/JavaScriptCore/builtins) 디렉토리에 [`ArrayPrototype.js`](https://github.com/WebKit/WebKit/blob/main/Source/JavaScriptCore/builtins/ArrayPrototype.js) 등 수많은 내장 JS가 들어있고, C++ 엔진 기능을 `@` 접두사 내부 함수로 호출함.
- **Firefox (SpiderMonkey)**: 엔진 내부 [builtin/](https://searchfox.org/mozilla-central/source/js/src/builtin/) 디렉토리에 [Array.js](https://searchfox.org/mozilla-central/source/js/src/builtin/Array.js) 등 표준 스펙을 JS로 구현한 **Self-hosted JavaScript**가 위치하며, C++ 엔진 기능은 [SelfHosting.cpp](https://searchfox.org/mozilla-central/source/js/src/vm/SelfHosting.cpp)의 특수 Intrinsic 함수로 호출함.
- **Node.js**: [`lib/internal/console/constructor.js`](https://github.com/nodejs/node/blob/main/lib/internal/console/constructor.js)에서 JS로 `Console`을 구현하고, 저수준 C++ 시스템 콜(`internalBinding('fs').writeBuffer`)을 호출함.
- **Chromium (V8)**: 과거에는 JS Builtin을 썼으나, 프로토타입 오염 방어와 JIT 성능을 위해 자체 정적 타입 DSL인 [**Torque**](https://v8.dev/docs/torque)를 개발하여 [`src/builtins/*.tq`](https://github.com/v8/v8/tree/main/src/builtins) (예: [`array-slice.tq`](https://github.com/v8/v8/blob/main/src/builtins/array-slice.tq)) 파일들을 초고속 기계어로 직접 컴파일함.

## 4. Handling Crashes

In [ ]:
import os
from browser import URL, Tab

# 1. 의도적으로 에러를 던지는 crash.js와 정상 동작하는 ok.js 생성
with open("crash.js", "w") as f:
    f.write('throw Error("Boom! 웹페이지 버그 발생");')

with open("ok.js", "w") as f:
    f.write('console.log("브라우저가 죽지 않고 후속 스크립트 실행 성공!");')

with open("crash_page.html", "w") as f:
    f.write('<script src="crash.js"></script><script src="ok.js"></script>')

# 2. 브라우저 탭에서 로드하여 예외 격리(Fault Isolation) 동작 검증
tab = Tab(600)
tab.load(URL("file://" + os.path.abspath("crash_page.html")))

# 3. 임시 파일 정리
os.remove("crash.js")
os.remove("ok.js")
os.remove("crash_page.html")

### 💡 [참고] 브라우저 크래시와 디버깅

#### 1. 웹페이지 스크립트 에러 vs 브라우저 런타임 에러
| 구분 | 웹페이지 스크립트 에러 (`crash.js`) | 브라우저 런타임 에러 (`runtime.js`, Python 함수) |
| :--- | :--- | :--- |
| **발생 주체** | 외부 웹 개발자 (신뢰할 수 없는 코드) | 브라우저 엔진 개발자 (브라우저 자체) |
| **처리 방침** | **격리(Fault Isolation)**: `try-except`로 잡고 브라우저는 생존 | **Fail-Fast**: 예외를 숨기지 않고 즉시 터뜨려 수정 |
| **숨겼을 때 위험** | 브라우저 프로세스가 강제 종료되어 모든 탭이 꺼짐 | DOM/이벤트가 조용히 먹통(Silent Failure)됨 |

#### 2. 크로스 콜스택 누락 방지
Python ↔ C(DukPy) ↔ JS 경계를 넘나들며 에러가 나면 스택 트레이스(Backtrace)가 중간에 잘리므로 양쪽에서 저장해야 함:
- **JS 쪽 (`runtime.js`)**: `try { ... } catch(e) { console.log(e.stack); throw e; }`
- **Python 쪽 (호스트 함수)**: `try: ... except: import traceback; traceback.print_exc(); raise`

## 5. Returning Handles

In [1]:
import os
from browser import URL, Tab

# 1. HTML 요소들과 querySelectorAll을 실행할 test.js 생성
with open("test.html", "w") as f:
    f.write('''
<div>First Div</div>
<div>Second Div</div>
<p>Paragraph</p>
<script src="test.js"></script>
''')

with open("test.js", "w") as f:
    f.write('''
var divs = document.querySelectorAll("div");
console.log("Found divs count: " + divs.length);
console.log("Div 0 handle: " + divs[0].handle);
console.log("Div 1 handle: " + divs[1].handle);
''')

# 2. 브라우저 탭에서 로드하여 핸들 발급 및 Node 객체 래핑 검증
tab = Tab(600)
tab.load(URL("file://" + os.path.abspath("test.html")))

# 3. 임시 파일 정리
os.remove("test.html")
os.remove("test.js")

Found divs count: 2
Div 0 handle: 0
Div 1 handle: 1


### 💡 [참고] 핸들(Handle) 시스템과 실제 브라우저의 DOM 바인딩

#### 1. 교재의 핸들 모델: 왜 정수 번호표를 쓸까?
- Python(호스트)과 JS 엔진(C-FFI)의 힙 메모리가 분리되어 있어 파이썬 객체를 직접 전달 불가 (`JSON serializable` 제약).
- OS 커널이 프로세스에 파일 포인터 대신 정수 파일 디스크립터(fd)를 주듯, 번호표(Handle)를 주고 파이썬 딕셔너리로 역참조함.

#### 2. 실제 브라우저(Chromium/WebKit)에서는 어떻게 동작할까?
교재의 `this.handle = handle`은 교육용 구현이며, 웹 표준 DOM에는 `handle` 속성이 없음. 실제 브라우저의 핵심 차이는 세 가지:

1. **포인터 은닉 (Internal Field)**:
   - 정수 번호 대신 V8 JS 객체 내부의 숨겨진 슬롯(`InternalField`)에 C++ `Element*` 주소를 저장함.
   - JS 문법(`Object.keys`, `Proxy` 등)으로는 절대 접근·변조할 수 없어 핸들 위조(Forging) 및 메모리 크래시 방지.
2. **객체 동일성 보장 (Wrapper Cache)**:
   - 교재는 `querySelectorAll`마다 `new Node(h)`를 만들어 `divs[0] !== divs[0]`이 됨.
   - 실제 브라우저는 `DOMDataStore`에 C++ 노드당 1개의 JS 래퍼만 캐싱해 동일성(`===`) 보장.
3. **크로스 힙 메모리 수거 (Unified GC)**:
   - 교재 방식은 `node_to_handle` 딕셔너리가 노드를 영구 보관해 메모리 누수 발생.
   - Chromium은 V8(JS 힙)과 Oilpan(C++ DOM 힙)이 서로의 참조를 교차 추적(Cross-Component Tracing)하여 순환 참조된 노드를 함께 해제.

## 6. Wrapping Handles

In [1]:
import os
from browser import URL, Tab

# 1. 100자 초과 텍스트를 검증할 input 요소들과 test.js 생성
# (주의: 우리 HTMLParser는 공백(whitespace) 기준으로 속성을 분리하므로 value에 공백 대신 언더스코어 사용)
with open("test.html", "w") as f:
    f.write('''
<input name="user_name" value="Short_text">
<input name="comment" value="This_is_a_very_long_comment_text_without_spaces_that_definitely_exceeds_one_hundred_characters_in_total_length_for_testing!">
<script src="test.js"></script>
''')

with open("test.js", "w") as f:
    f.write('''
var inputs = document.querySelectorAll("input");
for (var i = 0; i < inputs.length; i++) {
    var name = inputs[i].getAttribute("name");
    var value = inputs[i].getAttribute("value");
    if (value.length > 100) {
        console.log("Input " + name + " has too much text.");
    }
}
''')

# 2. 브라우저 탭에서 로드하여 getAttribute 메서드 동작 검증
tab = Tab(600)
tab.load(URL("file://" + os.path.abspath("test.html")))

# 3. 임시 파일 정리
os.remove("test.html")
os.remove("test.js")

Input comment has too much text.


### 💡 [참고] Handle Wrapping, 속성 반사(Reflection) 및 DOM 읽기 성능

#### 1. 역방향 데이터 흐름: JS Node 객체 → Python Element
- `querySelectorAll`이 Python → JS로 Handle 번호를 내려줬다면, `getAttribute`는 JS `Node` 인스턴스가 가진 `this.handle`을 다시 Python으로 전달하는 **역방향 호출** 구조임.
- JavaScript의 `Node.prototype.getAttribute`는 실질적인 로직 없이 `call_python("getAttribute", this.handle, attr)`을 호출하는 **RPC 프록시(Stub)** 역할만 수행함.

#### 2. HTML Attribute vs DOM Property (Reflection 메커니즘)
- **HTML Attribute (속성)**: 마크업 텍스트 파싱 결과로 저장되는 값 (`elt.attributes["value"]`). `getAttribute()` / `setAttribute()`로 접근.
- **DOM Property (프로퍼티)**: JavaScript 런타임 객체(`Node`)에 존재하는 필드 (`input.value`, `input.id`).
- **리플렉션 (Reflection)**: 브라우저는 자주 쓰는 속성(`id`, `src`, `href` 등)을 DOM 프로퍼티와 동기화(Reflect)함.
  - 현대 브라우저 엔진(V8, WebKit WebCore)은 이를 IDL(`interface HTMLElement`)에 정의된 `[Reflect]` 어노테이션으로 C++ 코드 생성기가 자동 바인딩함.
  - 우리 미니 브라우저에서는 프로퍼티 리플렉션 대신 `getAttribute` 메서드를 통해 파이썬 딕셔너리를 직접 조회함.

#### 3. DOM 속성 조회가 성능 저하를 유발하는 이유와 실제 브라우저 근거
1. **엔진 간 경계 이동 비용 (Bridge Crossing Overhead)**:
   - 단순 JS 변수 조회와 달리, `getAttribute` 호출 시 V8(JS)에서 Blink(C++) 렌더링 엔진으로 C-ABI 트램펄린 점프가 발생해 루프 순회 시 오버헤드 누적.
   - [WebKit 공식 기술 블로그 (Faster DOM Bindings)](https://webkit.org/blog/7397/faster-dom-bindings-in-webkit/) 및 Chromium Blink 문서에서 이 경계 호출 비용을 줄이기 위해 특수 JIT 인라이닝(DOM-JIT)을 연구해 옴.
2. **강제 동기 레이아웃 (Forced Synchronous Layout / Layout Thrashing)**:
   - 브라우저는 원래 렌더링 파이프라인(스타일 계산/Reflow)을 다음 프레임(16.6ms)까지 지연 일괄 처리(Lazy Batching)함.
   - 그러나 기하 속성(`offsetHeight`, `getBoundingClientRect` 등)을 읽는 순간, 최신 픽셀 수치를 반환하기 위해 대기 중인 스타일/레이아웃 계산을 즉시 강제 실행함.
   - **실제 Chromium 소스코드**: [`third_party/blink/renderer/core/dom/element.cc`](https://source.chromium.org/chromium/chromium/src/+/main:third_party/blink/renderer/core/dom/element.cc)의 `Element::OffsetHeight()`를 보면, 속성 반환 직전 `GetDocument().EnsurePaintLocationDataValidForNode(this)`를 호출하여 동기식 레이아웃 플러시(Reflow)를 강제하는 것을 확인할 수 있음.
   - **공식 가이드 & 레퍼런스**:
     - Google 공식 개발자 문서: [web.dev - Avoid large, complex layouts and layout thrashing](https://web.dev/articles/avoid-large-complex-layouts-and-layout-thrashing)
     - 레이아웃 강제 실행 속성 전수 조사: [What forces layout / reflow (by Paul Irish)](https://gist.github.com/paulirish/5d52fb082b5de975db3f)
   - 루프 내에서 읽기(`offsetHeight`)와 쓰기(`style.height = ...`)를 번갈아 호출하면 매 루프마다 브라우저 전체 트리 레이아웃이 재계산되어 프레임 드랍(Jank)이 발생하므로, 읽기 일괄 처리 후 쓰기 일괄 처리(Batching)로 해결해야 함.

## 7. Event Handling

In [1]:
import os
from browser import URL, Tab, Element, tree_to_list

# 1. keydown 이벤트 리스너를 등록하는 test.html / test.js 생성
with open("test.html", "w") as f:
    f.write('''
<input name="comment" value="12345">
<script src="test.js"></script>
''')

with open("test.js", "w") as f:
    f.write('''
function lengthCheck() {
    var name = this.getAttribute("name");
    var value = this.getAttribute("value");
    console.log("Keydown on " + name + ", current length=" + value.length);
    if (value.length > 10) {
        console.log("Input " + name + " has too much text.");
    }
}

var inputs = document.querySelectorAll("input");
for (var i = 0; i < inputs.length; i++) {
    inputs[i].addEventListener("keydown", lengthCheck);
}
''')

# 2. 브라우저 탭에서 로드 후 입력창 포커스 및 타이핑 시뮬레이션
tab = Tab(600)
tab.load(URL("file://" + os.path.abspath("test.html")))

inputs = [node for node in tree_to_list(tab.nodes, []) if isinstance(node, Element) and node.tag == "input"]
tab.focus = inputs[0]
tab.focus.is_focused = True

# 글자를 하나씩 입력할 때마다 keydown 이벤트가 디스패치됨
for char in "67890XY":
    tab.keypress(char)

# 3. 임시 파일 정리
os.remove("test.html")
os.remove("test.js")

Keydown on comment, current length=5
Keydown on comment, current length=6
Keydown on comment, current length=7
Keydown on comment, current length=8
Keydown on comment, current length=9
Keydown on comment, current length=10
Keydown on comment, current length=11
Input comment has too much text.


### 💡 [참고] 이벤트 디스패치 파이프라인과 핸들러 관리

#### 1. 호스트 → JS 역방향 호출 (제어의 역전, Inversion of Control)
- 지금까지는 JS가 Python 함수를 호출(`call_python`)하는 단방향 흐름이었으나, 이벤트는 OS/UI(사용자 입력)를 감지한 호스트(Python)가 JS 엔진을 역호출하는 구조임.
- `Tab.keypress` / `Tab.click` → `JSContext.dispatch_event` → `interp.evaljs(EVENT_DISPATCH_JS)` → `Node.prototype.dispatchEvent` 순으로 호출이 전파됨.

#### 2. `LISTENERS` 저장소의 위치와 `this` 바인딩
- **JS 힙에 저장하는 이유**: 이벤트 콜백은 클로저(Closure) 상태를 캡처하는 일급 함수이므로 Python으로 직렬화할 수 없음. 따라서 핸들별 리스너 맵(`LISTENERS[handle][type] = []`)은 JS 런타임에 보관함.
- **`list[i].call(this)`**: 콜백 실행 시 `call(this)`를 사용하여 함수 내부의 `this`가 이벤트를 수신한 `Node` 인스턴스를 정확히 가리키도록 명시적으로 바인딩함.

#### 3. 실제 브라우저(W3C DOM Level 2/3 Event Flow)와의 차이
- 교재는 이벤트가 발생한 대상(Target)에서만 즉시 콜백을 실행하는 단순 디스패치 방식임.
- 실제 브라우저는 [W3C DOM Event Architecture](https://www.w3.org/TR/DOM-Level-3-Events/#event-flow)에 따라 **3단계 전파(Capturing → Target → Bubbling)** 흐름을 지원하며, 최상위 `window`/`document`부터 타깃까지 DOM 트리를 순회하며 리스너를 호출함.